In [15]:
import pandas as pd

# Historical Tournament Results

df = pd.read_csv('/home/zburnside/pga_prediction_model/historical_tournament_results.csv')

    
# Weights for Course Profiles

WEIGHT_PROFILES = {
    "birdie_fest":   dict(finish=0.35, cut=0.15, early=0.35, sunday=0.15),
    "positional":    dict(finish=0.45, cut=0.25, early=0.20, sunday=0.10),
    "grinder":       dict(finish=0.45, cut=0.30, early=0.10, sunday=0.15),
    "major":         dict(finish=0.40, cut=0.30, early=0.10, sunday=0.20),
    "bomber":        dict(finish=0.40, cut=0.20, early=0.20, sunday=0.20),
    "volatile":      dict(finish=0.30, cut=0.20, early=0.30, sunday=0.20),
}

# Default weights for any tournament where the course switches

DEFAULT_WEIGHTS = dict(
    finish=0.45,     # avg_finish
    cut=0.25,        # cut_rate
    early=0.20,      # early_round_avg
    sunday=0.10,     # sunday_delta
)
                       
# Tournaments and their respective weights based on Course Profiles
TOURNAMENT_WEIGHTS = {

# ---------- BIRDIE FESTS ----------
"Sony Open in Hawaii": WEIGHT_PROFILES["birdie_fest"],
"The American Express": WEIGHT_PROFILES["birdie_fest"],
"John Deere Classic": WEIGHT_PROFILES["birdie_fest"],
"Rocket Mortgage Classic": WEIGHT_PROFILES["birdie_fest"],
"3M Open": WEIGHT_PROFILES["birdie_fest"],
"Shriners Children’s Open": WEIGHT_PROFILES["birdie_fest"],
"Sanderson Farms Championship": WEIGHT_PROFILES["birdie_fest"],
"Wyndham Championship": WEIGHT_PROFILES["birdie_fest"],
"Barracuda Championship": WEIGHT_PROFILES["birdie_fest"],

# ---------- POSITIONAL / ACCURACY COURSES ----------
"RBC Heritage": WEIGHT_PROFILES["positional"],
"Charles Schwab Challenge": WEIGHT_PROFILES["positional"],
"Valspar Championship": WEIGHT_PROFILES["positional"],
"Zurich Classic of New Orleans": WEIGHT_PROFILES["positional"],
"Travelers Championship": WEIGHT_PROFILES["positional"],
"FedEx St. Jude Championship": WEIGHT_PROFILES["positional"],
"BMW Championship": WEIGHT_PROFILES["positional"],

# ---------- GRINDERS ----------
"Arnold Palmer Invitational presented by Mastercard": WEIGHT_PROFILES["grinder"],
"Genesis Invitational": WEIGHT_PROFILES["grinder"],
"Memorial Tournament presented by Workday": WEIGHT_PROFILES["grinder"],
"Farmers Insurance Open": WEIGHT_PROFILES["grinder"],
"Mexico Open at Vidanta": WEIGHT_PROFILES["grinder"],
"Puerto Rico Open": WEIGHT_PROFILES["grinder"],

# ---------- BOMBER / DISTANCE TRACKS ----------
"The Players Championship": WEIGHT_PROFILES["bomber"],
"Wells Fargo Championship": WEIGHT_PROFILES["bomber"],
"Genesis Scottish Open": WEIGHT_PROFILES["bomber"],
"Zozo Championship": WEIGHT_PROFILES["bomber"],

# ---------- VOLATILE / WIND / LINKS ----------
"Open Championship": WEIGHT_PROFILES["volatile"],
"Genesis Scottish Open": WEIGHT_PROFILES["volatile"],
"RSM Classic": WEIGHT_PROFILES["volatile"],
"Butterfield Bermuda Championship": WEIGHT_PROFILES["volatile"],

# ---------- MAJORS ----------
"Masters Tournament": WEIGHT_PROFILES["major"],
"PGA Championship": WEIGHT_PROFILES["major"],
"U.S. Open": WEIGHT_PROFILES["major"],

# ---------- PLAYOFF / ELITE EVENTS ----------
"Tour Championship": WEIGHT_PROFILES["major"],
"FedEx St. Jude Championship": WEIGHT_PROFILES["major"],
"BMW Championship": WEIGHT_PROFILES["major"],
}

def tournament_course_history(df: pd.DataFrame, tournament_name: str, weights_map=None, default_weights=None) -> pd.DataFrame:
    weights_map = weights_map or {}
    w = (default_weights or DEFAULT_WEIGHTS).copy()
    w.update(weights_map.get(tournament_name, {}))  # override per tournament

    tdf = df[df["tournament_key"].eq(tournament_name)].copy()
    
    if tdf.empty:
        return pd.DataFrame()

    pos_raw = tdf["position"].astype(str).str.upper().str.strip()
    tdf["finish_pos"] = pos_raw.str.replace(r"[^0-9]", "", regex=True).replace("", pd.NA)
    tdf["finish_pos"] = pd.to_numeric(tdf["finish_pos"], errors="coerce")
    tdf["made_cut"] = ~pos_raw.isin(["CUT","WD","DQ","DNS"])

    for c in ["total","r1_score","r2_score","r3_score","r4_score"]:
        if c in tdf.columns:
            tdf[c] = pd.to_numeric(tdf[c], errors="coerce")

    player_hist = (
        tdf.groupby("playerName")
        .agg(
            starts=("position","count"),
            cut_rate=("made_cut","mean"),
            avg_finish=("finish_pos","mean"),
            avg_r1=("r1_score","mean"),
            avg_r2=("r2_score","mean"),
            avg_r4=("r4_score","mean"),
        )
        .reset_index()
    )

    player_hist["early_round_avg"] = player_hist[["avg_r1","avg_r2"]].mean(axis=1)
    player_hist["sunday_delta"] = player_hist["avg_r4"] - player_hist["avg_r1"]

    player_hist["course_fit_score"] = (
        (100 - player_hist["avg_finish"]) * w["finish"] +
        (player_hist["cut_rate"] * 100)  * w["cut"] +
        (70 - player_hist["early_round_avg"]) * w["early"] +
        (5 - player_hist["sunday_delta"]) * w["sunday"]
    )

    return (player_hist[player_hist["starts"] >= 2]
            .sort_values("course_fit_score", ascending=False)
            .reset_index(drop=True))


/tmp/ipykernel_29/334110695.py:5: DtypeWarning: Columns (9,18,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/zburnside/pga_prediction_model/historical_tournament_results.csv')


In [16]:
sony = tournament_course_history(df, "Sony Open in Hawaii",
                                 weights_map=TOURNAMENT_WEIGHTS)


In [18]:
amex = tournament_course_history(df, "The American Express",
                                 weights_map=TOURNAMENT_WEIGHTS)
amex

,playerName,starts,cut_rate,avg_finish,avg_r1,avg_r2,avg_r4,early_round_avg,sunday_delta,course_fit_score
0,Justin Thomas,2,1.0,2.500000,66.0,65.500000,67.000000,65.750000,1.000000,51.2125
1,Xander Schauffele,2,1.0,3.000000,64.5,68.500000,63.500000,66.500000,-1.000000,51.0750
2,Jon Rahm,2,1.0,7.500000,65.0,67.000000,69.500000,66.000000,4.500000,48.8500
3,Paul Casey,2,1.0,14.500000,70.0,66.000000,70.000000,68.000000,0.000000,46.3750
4,Abraham Ancer,3,1.0,15.666667,68.0,67.666667,66.333333,67.833333,-1.666667,46.2750
...,...,...,...,...,...,...,...,...,...,...
245,Taylor Pendrith,3,0.0,NaN,71.0,70.333333,NaN,70.666667,NaN,NaN
246,Trevor Cone,2,0.0,NaN,68.0,73.500000,NaN,70.750000,NaN,NaN
247,Tyson Alexander,2,0.0,NaN,74.0,73.000000,NaN,73.500000,NaN,NaN
248,Wesley Bryan,3,0.0,NaN,70.0,72.666667,NaN,71.333333,NaN,NaN
